# 静态面板门槛模型：理论与方法

**Keywords**: 面板门槛模型, Hansen, xthreg, 非线性关系, Bootstrap, 门槛效应检验


<!-- ## 目录

1. [简介](#1-简介)
2. [单一门槛模型](#2-单一门槛模型)
3. [多重门槛模型](#3-多重门槛模型)
4. [Stata 实操：xthreg 命令](#4-stata-实操xthreg-命令)
5. [模拟数据实例](#5-模拟数据实例)
6. [实证应用案例](#6-实证应用案例)
7. [参考文献](#7-参考文献) -->

## 简介

<!-- 自从 Tong (1978) 提出门限自回归模型 (Threshold Auto-regression, 简称 TAR) 后，这种非线性时间序列模型在经济和金融领域得到了广泛的应用。虽然 TAR 模型大部分被应用于时间序列资料,但 Tiao and Tsay (1994)、Potter (1995)、Marterns et al. (1998) 也利用此方法分析横截面资料或面板资料。 -->



### 什么是门槛效应？

在经济学研究中，变量之间的关系往往不是简单的线性关系。当某个关键变量超过某个临界点时，其他变量对因变量的影响可能会发生显著变化。这就是所谓的**门槛效应 (threshold effect)**。

**典型的经济现象：**

- 人力资本与经济增长：人力资本积累到一定程度后，对经济增长的促进作用会显著增强
- 第一大股东持股比例与公司价值：存在倒 U 型关系
- 资本结构与公司价值：在最优资本结构附近存在门槛效应
- 债务与经济增长：政府债务在某个阈值之后对经济增长产生负面影响
- 通胀与经济增长：通货膨胀率超过某个阈值后对经济增长的负面影响显著

### 传统处理方法的局限

传统的处理非线性关系的方法包括：

1. **加入二次项**：`reg y x x_sq`
   - 问题：$x$ 与 $x^2$ 往往高度共线
   
2. **加入虚拟变量和交乘项**：`reg y x d dx`
   - 问题：如何确定分组界点？错误的界点会导致严重的偏误
   
3. **分组回归**
   - 问题：分组标准主观性强，缺乏统计检验支持

### 面板门槛模型的优势

门限自回归模型在计量方法上提供了更为客观的研究方式，利用门限变量 (threshold variable) 来决定不同的分界点，进而利用门限变量的观察值估计出适合的门限值。这可以有效避免一般研究者所使用的主观判定分界点法所造成的偏误。

**Hansen (1999) 的主要贡献：**

- 提出了估计门槛值 $\gamma$ 的最小二乘法
- 解决了门槛效应检验中的 "Davies Problem"
- 通过 Bootstrap 方法构造检验统计量的渐进分布
- 利用似然比检验构造门槛值的置信区间

### 广泛的实证应用

面板门槛模型在经济学的多个主题中有广泛的应用：

- **债务与经济增长**：Adam and Bevan (2005)、Cecchetti et al. (2011)、Chudik et al. (2017)
- **通胀与经济增长**：Khan and Ssnhadji (2001)、Rousseau and Wachtel (2002)、Kremer et al. (2013)
- **FDI 与企业生产率**：Girma (2005)
- **资本结构调整**：Dang et al. (2012) 使用动态面板门槛模型

## 单一门槛模型

### 模型的基本设定

基本模型设定如下：

$$
y_{it} = \mu_i + x_{it}' \beta_1 \cdot I(q_{it} \le \gamma) + x_{it}' \beta_2 \cdot I(q_{it} > \gamma) + \varepsilon_{it} \tag{1}
$$

其中：

- $i = 1, 2, \cdots, N$ 表示不同的个体
- $t = 1, 2, \cdots, T$ 表示时间
- $q_{it}$ 为门槛变量
- $y_{it}$ 为被解释变量
- $x_{it}$ 为解释变量向量
- $I(\cdot)$ 为指标函数，相应条件成立时取值为 1，否则取值为 0
- $\mu_i$ 为个体固定效应
- $\gamma$ 为门槛值（未知参数）
- $\beta_1, \beta_2$ 为回归系数向量
- $\varepsilon_{it}$ 为误差项，假设 $\varepsilon_{it} \sim i.i.d \ N(0, \sigma^2)$

**用分段函数表示更为清晰：**

$$
y_{it} = \begin{cases}
\mu_i + x_{it}' \beta_1 + \varepsilon_{it} & \text{若 } q_{it} \le \gamma \\
\mu_i + x_{it}' \beta_2 + \varepsilon_{it} & \text{若 } q_{it} > \gamma
\end{cases} \tag{2}
$$

**模型的经济含义：**

依据门限变量 $q_{it}$ 与门限值 $\gamma$ 的相对大小，我们可以将样本观察值分成两个区间 (regime)。区间的差异表现在回归系数 $\beta_1$ 和 $\beta_2$ 的不同上。

**重要假设：**

1. 为保证 $\beta_1$ 和 $\beta_2$ 可以识别，$x_{it}$ 中不能包含不随时间改变的变量（如性别、国籍等）
2. 门限变量 $q_{it}$ 可以随时间变化
3. $\varepsilon_{it}$ 服从 i.i.d 假设，意味着模型是静态的，不能包含被解释变量的滞后项

### 模型的估计

#### 第一步：去除个体固定效应

采用组内去均值 (within transformation) 的方法去除个体效应 $\mu_i$。这与处理一般固定效应模型的方法相同。

对 (1) 式中所有时期取组内平均得到：

$$
\bar{y}_i = \mu_i + \bar{x}_i(\gamma)' \beta + \bar{\varepsilon}_i \tag{3}
$$

其中：

$$
\bar{y}_i = \frac{1}{T} \sum_{t=1}^T y_{it}, \quad
\bar{\varepsilon}_i = \frac{1}{T} \sum_{t=1}^T \varepsilon_{it}
$$

$$
\bar{x}_i(\gamma) = \begin{pmatrix}
\frac{1}{T} \sum_{t=1}^T x_{it} I(q_{it} \le \gamma) \\
\frac{1}{T} \sum_{t=1}^T x_{it} I(q_{it} > \gamma)
\end{pmatrix}
$$

用 (1) 式减去 (3) 式，得到去均值后的模型：

$$
y_{it}^* = x_{it}^*(\gamma)' \beta + \varepsilon_{it}^* \tag{4}
$$

其中 $y_{it}^* = y_{it} - \bar{y}_i$，$x_{it}^*(\gamma) = x_{it}(\gamma) - \bar{x}_i(\gamma)$，$\varepsilon_{it}^* = \varepsilon_{it} - \bar{\varepsilon}_i$。

#### 第二步：估计参数

对于给定的 $\gamma$ 值，可以采用普通最小二乘法 (OLS) 得到参数 $\beta$ 的一致估计量：

$$
\hat{\beta}(\gamma) = [X^*(\gamma)' X^*(\gamma)]^{-1} X^*(\gamma)' Y^* \tag{5}
$$

相应的残差向量为：

$$
\hat{e}^*(\gamma) = Y^* - X^*(\gamma) \hat{\beta}(\gamma) \tag{6}
$$

残差平方和为：

$$
S_1(\gamma) = \hat{e}^*(\gamma)' \hat{e}^*(\gamma) \tag{7}
$$

#### 第三步：网格搜索门槛值

Chan (1993) 和 Hansen (1999) 建议采用最小二乘法来估计 $\gamma$。通过最小化残差平方和来获得 $\gamma$ 的估计值：

$$
\hat{\gamma} = \arg\min_{\gamma} S_1(\gamma) \tag{8}
$$

一旦得到了 $\hat{\gamma}$，便可进而得到：

- $\hat{\beta} = \hat{\beta}(\hat{\gamma})$：系数估计值
- $\hat{e}^* = \hat{e}^*(\hat{\gamma})$：残差向量
- $\hat{\sigma}^2 = \frac{1}{n(T-1)} \hat{e}^{*'} \hat{e}^* = \frac{1}{n(T-1)} S_1(\hat{\gamma})$：残差方差

### 计算相关问题

在对 (8) 式极小化从而得到门限值 $\gamma$ 之 OLS 估计量的过程中，由于残差平方和 $S_1(\gamma)$ 仅决定于 $\gamma$，所以它是 $\gamma$ 的一个非连续函数，其中至多有 $nT$ 个间断点（因为样本容量为 $nT$）。但事实上，我们在极小化 (8) 式的过程中，仅需要对那些**非重复的门槛值** $q_{it}$ 进行搜索即可。

#### 网格搜索步骤：

**第一步**：对门限变量 $q_{it}$ 中非重复的观察值进行排序，去掉其中最大和最小的观察值各 $\eta\%$ 个（$\eta > 0$）。

**第二步**：用余下的 $N^*$ 个观察值作为估计样本，将 $q_{it}$ 值从小到大依次代入 (4) 式进行估计，得到相应的残差平方和 (7)，后者的最小值便对应着 $\gamma$ 的估计值 $\hat{\gamma}$。

#### 加速搜索的技巧：

如果 $N^*$ 是一个很大的数值，上面介绍的方法可能相当费时。我们可以采用一种更为简洁的等价方式：

**不必搜索所有介于第 $\eta\%$ 和 $(1-\eta)\%$ 百分位之间的 $q_{it}$ 值，而是仅仅搜索特定百分位上的数值。**

例如，按照 {1.00%, 1.25%, 1.50%, 1.75%, 2%, $\cdots$, 99.0%} 栅格进行搜索，共包括 393 个分位值。采用这种方法在多数实证分析中都能达到我们所期望的精度。

**Hansen (1999, p.349) 的建议：**

- 只搜索门槛变量 $q_{it}$ 中的非重复值
- 对这些非重复值排序，在 [1th, 99th] 区间内搜索
- 如果需要搜索的格点仍然很多，可以在上述区间内仅搜索特定的百分位数

### 假设检验

#### 门槛效应的检验

虽然在模型设定中假设存在门限效应，但其是否具有统计上的显著性，还需要做进一步的检验。

**原假设**：不存在门限效应

$$
H_0: \beta_1 = \beta_2
$$

**备择假设**：

$$
H_1: \beta_1 \ne \beta_2
$$

**Davies Problem：**

在原假设 $H_0$ 下，门限值 $\gamma$ 是无法识别的，此时传统检验统计量的分布是非标准的。这就是所谓的 "Davies Problem"。

因为在原假设 $H_0$ 下，我们事实上对 (1) 式施加了线性约束 $\beta_1 - \beta_2 = 0$，所以不存在一个唯一的 $\gamma$ 值使 (1) 式成立。

#### 似然比检验统计量

在不存在门限效应的原假设下，模型 (1) 转化为：

$$
y_{it} = \mu_i + x_{it}' \beta_1 + \varepsilon_{it} \tag{9}
$$

去除个体效应后：

$$
y_{it}^* = x_{it}^* \beta_1 + \varepsilon_{it}^* \tag{10}
$$

采用 OLS 得到 $\beta_1$ 的估计量 $\tilde{\beta}_1$，相应的残差为 $\tilde{e}_{it}^*$，残差平方和为 $S_0 = \tilde{e}^{*'} \tilde{e}^*$。

似然比检验统计量为：

$$
F_1 = \frac{S_0 - S_1(\hat{\gamma})}{\sigma^2} = \frac{S_0 - S_1(\hat{\gamma})}{S_1(\hat{\gamma}) / [n(T-1)]} \tag{11}
$$

$F_1$ 的渐进分布是非标准的，完全不同于卡方分布。而且，一般而言其分布依赖于样本的矩（如均值、方差、峰度和偏度等），所以临界值无法查表得到。

#### Bootstrap 方法

Hansen (1996) 研究表明，采用 "自体抽样法" (Bootstrap) 可以获得其一阶渐进分布，基于此构造的 p 值也将是渐进有效的。

**Bootstrap 抽样步骤：**

**第一步**：在反复抽样过程中，假设解释变量 $x_{it}$ 和门槛变量 $q_{it}$ 都是固定不变的。将估计备择假设对应的模型 (4) 得到的残差 $\hat{e}^*$ 按个体分组：

$$
\hat{e}_i^* = (\hat{e}_{i1}^*, \hat{e}_{i2}^*, \cdots, \hat{e}_{iT}^*)
$$

把由此得到的样本观察值 $\hat{e}_1^*, \hat{e}_2^*, \cdots, \hat{e}_n^*$ 视为 "自体抽样" 的实证分布 (empirical distribution)。

**第二步**：从实证分布中随机抽取（可重复）$n$ 个样本观察值，构造出原假设 $H_0$ 下的 "自体抽样" 样本：

$$
Y^{bs} = X^*(\gamma) \beta + e^{bs}
$$

**第三步**：利用第二步构造的 "自体抽样" 样本分别估计原假设对应的模型 (10) 和备择假设对应的模型 (4)，计算由 (11) 式决定的似然比统计量。

**第四步**：重复以上过程多次（如 500 次），计算模拟值大于真实值的概率，这便是我们采用 "自体抽样" 法得到的原假设 $H_0$ 下 $F_1$ 统计量的渐进 p 值。

如果得到的 p 值小于我们设定的临界值（如 5%），那么就拒绝原假设，从而认为存在门槛效应。

**注意事项：**

需要说明的是，在原假设 $H_0$ 下，$F_1$ 统计量与参数 $\beta_1$ 无关，所以任何的 $\beta_1$ 值都可以使用。

#### 门槛估计值的渐进分布

在已经确认存在门槛效应的情况下（$\beta_1 \ne \beta_2$），Chan (1993) 以及 Hansen (1999) 研究表明 $\hat{\gamma}$ 是 $\gamma_0$（$\gamma$ 的真实值）的一致估计量，然而其渐进分布是高度非标准的。

Hansen (1999) 指出，构造 $\gamma$ 的置信区间的最佳方法是利用似然比统计量构造出 "非拒绝域"。

**似然比统计量：**

对于原假设 $H_0: \gamma = \gamma_0$ 而言，似然比统计量为：

$$
LR_1(\gamma) = \frac{S_1(\gamma) - S_1(\hat{\gamma})}{\hat{\sigma}^2} \tag{12}
$$

如果 $LR_1(\gamma_0)$ 的值足够大，那么我们就拒绝原假设。

**注意**：(12) 式的统计量和 (11) 式的统计量所检验的原假设是不同的。$LR_1(\gamma_0)$ 用于检验 $H_0: \gamma = \gamma_0$，而 $F_1$ 用于检验 $H_0: \beta_1 = \beta_2$。

**渐进分布：**

在满足一系列假设条件及 $H_0: \gamma = \gamma_0$ 的情况下，Hansen (1999) 导出 $LR_1(\gamma)$ 的渐进分布：

$$
LR_1(\gamma) \xrightarrow{d} \xi \quad (n \to \infty) \tag{13}
$$

其中，$\xi$ 是一个具有如下分布函数的随机变量：

$$
P(\xi \le x) = (1 - \exp(-x/2))^2 \tag{14}
$$

这表明似然比统计量具有非标准分布，但**不存在 "未知参数" 问题** (free of nuisance parameters)。由于分布函数 (14) 的渐进分布是枢轴的 (pivotal)，因此可用来构造一个有效渐进的置信区间。

分配函数 (14) 的反函数可表示为：

$$
c(\alpha) = -2 \ln(1 - \sqrt{1-\alpha}) \tag{15}
$$

据此我们可以很方便地计算出临界值：

- 10% 显著水平：6.53
- 5% 显著水平：7.35
- 1% 显著水平：10.59

如果 $LR_1(\gamma_0)$ 的值大于 $c(\alpha)$，那么我们就在 $\alpha$ 显著水平上拒绝原假设 $H_0: \gamma = \gamma_0$。

**置信区间的构造：**

在 $1-\alpha$ 置信水平上的 "非拒绝域" 是指一系列满足 $LR_1(\gamma) \le c(\alpha)$ 的 $\gamma$ 值。作为一种简易的直观判断方法，我们可以绘制出以 $LR_1(\gamma)$ 为纵坐标、$\gamma$ 为横坐标的二维图，同时在 $c(\alpha)$ 处画一条水平线，以确定置信区间。

#### 系数估计值的渐进分布

估计值 $\hat{\beta} = \hat{\beta}(\hat{\gamma})$ 与门槛估计值 $\hat{\gamma}$ 相关联，这似乎使得对 $\beta$ 的统计推论变得相当复杂。Chan (1993) 和 Hansen (1999) 指出这种相关性并不会显著影响一阶渐进性质，所以对 $\beta$ 的统计推论可以在假设门槛估计值 $\hat{\gamma}$ 就是其真实值的情况下进行。

因此，$\hat{\beta}$ 的渐进分布将是**正态分布**，其协方差矩阵可以用下式估计得到：

$$
\hat{V} = \hat{\sigma}^2 \left( \sum_{i=1}^n \sum_{t=1}^T x_{it}^*(\hat{\gamma}) x_{it}^*(\hat{\gamma})' \right)^{-1} \tag{16}
$$

虽然在构造门槛 $\gamma$ 的置信区间时，我们需要假设残差项为独立同分布的 (i.i.d)，但是在此处构造估计系数的置信区间时，我们完全可以**放松这个假设**。

**异方差稳健的标准误：**

在允许残差项存在条件异方差的情况下，$\hat{\beta}$ 的方差-协方差矩阵的估计式为：

$$
\hat{V}_h = A^{-1} B A^{-1} \tag{17}
$$

其中：

$$
A = \sum_{i=1}^n \sum_{t=1}^T x_{it}^*(\hat{\gamma}) x_{it}^*(\hat{\gamma})'
$$

$$
B = \sum_{i=1}^n \sum_{t=1}^T x_{it}^*(\hat{\gamma}) x_{it}^*(\hat{\gamma})' (\hat{e}_{it}^*)^2
$$

这事实上是 White (1980) 估计式。

## 多重门槛模型

模型 (1) 中仅有一个门槛，但在许多情况下门槛的个数可能不止一个。

### 双重门槛模型的设定

双重门槛模型的设定如下：

$$
\begin{aligned}
y_{it} = & \mu_i + x_{it}' \beta_1 \cdot I(q_{it} \le \gamma_1) \\
       & + x_{it}' \beta_2 \cdot I(\gamma_1 < q_{it} \le \gamma_2) \\
       & + x_{it}' \beta_3 \cdot I(q_{it} > \gamma_2) + \varepsilon_{it}
\end{aligned} \tag{18}
$$

其中 $\gamma_1 < \gamma_2$。这里我们集中讨论双重门槛模型，因为由此可以很方便地扩展到更多门槛的情形。

### 估计方法

对于给定的 $(\gamma_1, \gamma_2)$，(18) 式是关于 $(\beta_1, \beta_2, \beta_3)$ 的线性模型，所以我们可以用 OLS 进行估计。因此，对于每一组给定的 $(\gamma_1, \gamma_2)$，我们可以首先计算出相应的残差平方和 $S(\gamma_1, \gamma_2)$，这与我们对单一门槛模型的处理相似。

$(\gamma_1, \gamma_2)$ 的联合 OLS 估计值就是使得 $S(\gamma_1, \gamma_2)$ 最小时对应的值 $(\hat{\gamma}_1, \hat{\gamma}_2)$。虽然从理论上讲，同时估计 $(\hat{\gamma}_1, \hat{\gamma}_2)$ 是可行的，但是实际操作的过程中这种做法是非常费时的。我们对 $(\gamma_1, \gamma_2)$ 进行逐个搜索大约需要进行 $N^2 = (nT)^2$ 次回归。

#### 循环法 (Sequential Method)

为了减小运算量，我们采用 "循环法" 进行估计。在含有多个结构突变点的模型中，这种方法可以得到参数的一致估计量，如 Chong (1994)、Bai (1997) 以及 Bai and Perron (1998)。

**第一步**：设 $S_1(\gamma)$ 为由 (7) 式所定义的单一门槛设定下的残差平方和，$\hat{\gamma}_1$ 为使得 $S_1(\gamma)$ 最小时对应的门槛估计值。

Chong (1994) 以及 Bai (1997) 的研究表明无论对于 $\gamma_1$ 还是 $\gamma_2$ 而言，$\hat{\gamma}_1$ 都是 $\gamma_1$ 的一致估计量。

**第二步**：固定第一步得到的 $\hat{\gamma}_1$，估计 (18) 式，则第二步的筛选标准为：

$$
S^{\gamma_2}(\gamma_2) = \begin{cases}
S(\hat{\gamma}_1, \gamma_2) & \text{若 } \hat{\gamma}_1 < \gamma_2 \\
S(\gamma_2, \hat{\gamma}_1) & \text{若 } \gamma_2 < \hat{\gamma}_1
\end{cases} \tag{19}
$$

进而得到第二步的门槛估计值为：

$$
\hat{\gamma}_2^{\gamma} = \arg\min_{\gamma_2} S^{\gamma_2}(\gamma_2) \tag{20}
$$

由于若个别区间内的观察值过少我们无法进行有效的估计，所以我们可以在搜索 (19) 式的过程中限制每个区间内的最小观察值个数。

Bai (1997) 的研究表明 $\hat{\gamma}_2^{\gamma}$ 是渐进有效的，但 $\hat{\gamma}_1$ 的估计却不具有此性质。这是因为在估计 $\hat{\gamma}_1$ 的过程中，残差平方和中包含了我们所忽略的区间造成的影响。

**第三步**：固定 $\hat{\gamma}_2^{\gamma}$，重新估计 $\gamma_1$，此时参数的筛选标准为：

$$
S^{\gamma_1}(\gamma_1) = \begin{cases}
S(\gamma_1, \hat{\gamma}_2^{\gamma}) & \text{若 } \gamma_1 < \hat{\gamma}_2^{\gamma} \\
S(\hat{\gamma}_2^{\gamma}, \gamma_1) & \text{若 } \hat{\gamma}_2^{\gamma} < \gamma_1
\end{cases} \tag{21}
$$

得到更新后的 $\hat{\gamma}_1$ 的估计值为：

$$
\hat{\gamma}_1^{\gamma} = \arg\min_{\gamma_1} S^{\gamma_1}(\gamma_1) \tag{22}
$$

Bai (1997) 指出在结构突变模型中，更新后的估计值 $\hat{\gamma}_1^{\gamma}$ 是渐进有效的，所以我们认为在门槛模型中这个结论仍然成立。

### 门槛个数的确定

在模型 (18) 的设定中，可能不存在门槛，也可能存在一个或两个门槛值，因此我们需要对此进行检验。

#### 检验单一门槛 vs 双重门槛

如果第一步的 $F_1$ 拒绝了原假设，即存在一个门槛，那么在模型 (18) 的设定中，我们就需要作进一步的检验以便区分单一门槛和双重门槛。

我们从第二步中估计出的最小残差平方和为 $\hat{\sigma}^2 = S^{\gamma_2} / [n(T-1)]$，因此我们可以基于下面的统计量来检验单一门槛与双重门槛哪个更为显著：

$$
F_2 = \frac{S_1(\hat{\gamma}_1) - S^{\gamma_2}(\hat{\gamma}_2^{\gamma})}{\hat{\sigma}^2} \tag{23}
$$

该检验量事实上是一个近似的似然比检验。如果 $F_2$ 的值较大，那么我们就拒绝仅存在一个门槛值的原假设。

由于在原假设下，似然比统计检验量的渐进分布是非枢轴的 (non-pivotal)，Hansen (1999) 建议采用 "自体抽样法" 来近似其样本分布。抽样方法和检验统计量的构造方法与我们在前面介绍的 $F_1$ 统计量的构造方法相似。

区别在于，此时我们的备择假设对应的模型为 (18) 式，而原假设对应的模型为 (1)。因此，我们产生的自抽样样本为：

$$
y_{it}^{\#} = x_{it}' \hat{\beta}_1 \cdot I(q_{it} \le \hat{\gamma}) + x_{it}' \hat{\beta}_2 \cdot I(q_{it} > \hat{\gamma}) + \varepsilon_{it}^{\#} \tag{24}
$$

从 (24) 式我们可以看出，$F_2$ 的抽样分布依赖于回归参数 $\beta_1 - \beta_2$ 和 $\gamma$。这不同于 $LR_1$ 的抽样分布，$LR_1$ 并不依赖于任何回归参数。所以，$F_2$ 的抽样分布是非枢轴的，使得我们无法保证 "自体抽样法" 能产生很好的近似效果。

### 置信区间的构造

Bai (1997) 研究表明，我们在 3.2 节第三步得到的更新后的 $\hat{\gamma}_1$ 的估计值与在单一门槛模型中得到的估计值具有相同的渐进分布。这启发我们可以采用与 2.4.3 节相同的方法来构造置信区间。

设：

$$
LR^{\gamma_2}(\gamma) = \frac{S^{\gamma_2}(\gamma) - S^{\gamma_2}(\hat{\gamma}_2^{\gamma})}{\hat{\sigma}^2} \tag{25}
$$

$$
LR^{\gamma_1}(\gamma) = \frac{S^{\gamma_1}(\gamma) - S^{\gamma_1}(\hat{\gamma}_1^{\gamma})}{\hat{\sigma}^2} \tag{26}
$$

其中，$S^{\gamma_2}(\gamma)$ 和 $S^{\gamma_1}(\gamma)$ 分别由 (19) 和 (21) 式定义。

$\gamma_2$ 和 $\gamma_1$ 在 $(1-\alpha)\%$ 显著水平上的置信区间分别为一系列满足 $LR^{\gamma_2}(\gamma) \le c(\alpha)$ 和 $LR^{\gamma_1}(\gamma) \le c(\alpha)$ 的 $\gamma$ 值。

## 参考文献

- Hansen, B. E. (**1999**). Threshold effects in non-dynamic panels: Estimation, testing, and inference. **Journal of Econometrics**, 93(2), 345–368. [Link](https://doi.org/10.1016/S0304-4076(99)00025-1), [-PDF-](http://sci-hub.ren/10.1016/S0304-4076(99)00025-1), [Google](https://scholar.google.com/scholar?q=Threshold+effects+in+non-dynamic+panels).
- Wang, Q. **2015**. Fixed-Effect Panel Threshold Model using Stata. **The Stata Journal**, 15(1), 121–134. [Link](https://journals.sagepub.com/doi/10.1177/1536867X1501500108), [PDF](https://journals.sagepub.com/doi/pdf/10.1177/1536867X1501500108), [Google](<https://scholar.google.com/scholar?q=Fixed-Effect Panel Threshold Model using Stata>).

